# Get data needed for aok

In [ ]:
from datetime import datetime, timedelta
import time

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
# import rasterio as rio
# from rasterio.session import AWSSession
from shapely.geometry import Polygon
import xarray as xr

import boto3
import earthaccess
import icepyx as ipx
from sliderule import sliderule, icesat2 #, io

from aok.core.acquisition.base import DataRequest # data request object


### Specify the test data

Options for specifying input data:
1. manually
2. from an example "test_site", where the relevant information is stored in the `test_sites.yaml` file.

In [ ]:
# 1. Manually specify inputs

spatial_extent = [-77.54, 36.66, -74.58, 39.63] # chesapeak bay
short_name = 'ATL03'
temporal = [datetime(2018, 10, 22, 6),datetime(2018, 10, 26, 18)]


In [ ]:
# 2. Grab inputs from test file

# TODO: add typing to these functions and the code that grabs the test cases, below.

import yaml
from pathlib import Path
from shapely.geometry import Point

def load_test_sites(path="./test_sites.yaml"):
    with Path(path).open("r") as f:
        return yaml.safe_load(f)

def get_region_by_name(name, sites=None):
    if sites is None:
        sites = load_test_sites()
    for site in sites["locations"]:
        if site["name"] == name:
            return site
    raise KeyError(f"Region not found: {name}")

def check_not_null(key):
    if key is None or all(l is None for l in key):
        return False
    else:
        return True

def get_bbox_shapely(lat, lon, buffer_deg) -> list:
    point = Point(lon, lat)
    # Creating a 'square' buffer
    bbox_poly = point.buffer(buffer_deg, cap_style=3) 
    return bbox_poly.bounds  # Returns (min_lon, min_lat, max_lon, max_lat)



In [ ]:
site = get_region_by_name("chesapeake_bay")

spatial = site["spatial_extent"]
if check_not_null(spatial["bbox"]):
    spatial_extent = spatial["bbox"]
elif check_not_null(spatial["latlon"]):
    spatial_extent = get_bbox_shapely(spatial["latlon"][0],
                                      spatial["latlon"][1],
                                      spatial["buffer"])
else:
    raise ValueError("Missing spatial extent")

if any([check_not_null(site["dates"]["start"]), check_not_null(site["dates"]["end"])]):
    
    temporal = [
        datetime.fromisoformat(site["dates"]["start"]),
        datetime.fromisoformat(site["dates"]["end"]),
    ]
else:
    raise ValueError("Missing temporal inputs")

In [ ]:
print(site)

In [ ]:
# extract region from bounding box
srregion = sliderule.toregion(source = site["spatial_extent"]["bbox"], )
print(srregion)

In [ ]:
srregion["raster"]

## Get data with SlideRule

In [ ]:
# Initialize SlideRule client
sliderule.init("slideruleearth.io")

## Atl03 request

In [ ]:
# Build ATL03 Request
# Needed parameters
parms = {
    "poly": srregion["poly"], #srextent,
    "t0": f'{temporal[0]:%Y-%m-%dT%H:%M:%SZ}',
    "t1": f'{temporal[1]:%Y-%m-%dT%H:%M:%SZ}',
    "srt": [0,1,2,3,4],
    "cnf": [-2, -1, 0, 1, 2, 3, 4], # confidence
    #"atl24_class":[0, 40, 41],  
    "atl03_ph_fields": ["h_ph",
                        "delta_time", 
                        "dist_ph_along",
                       ], # hieghts parameters
    "atl03_geo_fields": ["segment_id",
        "ph_index_beg",
        "segment_ph_cnt",
        "segment_dist_x", # "Equator_Segment_Distance"
        "segment_length",
        #"delta_time", # "delta_time" in data_processing # ERROR DUE TO MULTIPLE DELTA TIMES
        "reference_photon_lat", # used to check if segment is on land or not
        "reference_photon_lon", # used to check if segment is on land or not
        "ref_elev", # used in interpolation to find ph_ref_elev
        "ref_azimuth", # interpollated and passed along as ph_ref_azimuth
    ],
    "atl03_bckgrd_fields": [ # used to calculate photon_background rate.
        "bckgrd_rate", 
        #"delta_time", # called "bckgrd_time" 
        ],
       "atl03_cor_fields": ["geoid"], # height above WGS-84 ref ellipsoid
    "output": {
        "path": "./test_data/kdOutputAsGeo.geoparquet",
        "format": "parquet",
        "as_geo": True,
        "open_on_complete": True
    }
}

In [ ]:
# Build the parameters for sliderule request to atl03
sliderule_req = DataRequest(spatial=srregion["poly"],
                    date_range=(f'{temporal[0]:%Y-%m-%d}', f'{temporal[1]:%Y-%m-%d}'), need_atl24 = False,
                    download_dir= "./test_data/")
sliderule_req

In [ ]:
sr_acquisition = sliderule_req.get_sliderule_data()
sr_acquisition

In [ ]:
atl24_sr = sliderule_req.get_atl24_data()

In [ ]:
list(atl24_sr.photons)

In [ ]:
list(all_sr)

In [ ]:
 sliderule.earthdata.search(parms, resources=None)

In [ ]:
all_sr[0].photons

In [ ]:
full_results = all_sr[0].photons.merge(all_sr[1].segments, 
                left_on = "time_ns", 
                right_on = "time_ns", 
                how = "left", 
                suffixes = ("_atl03", "_atl24"))

In [ ]:
all_sr[2]

In [ ]:
# Read the output file into a GeoDataFrame
output_file = sliderule.run("atl03x", parms)

In [ ]:
output_file.info()

In [ ]:
output_file.head()

In [ ]:
output_file["bckgrd_rate"]

In [ ]:
gdf = gpd.read_parquet(parms["output"]["path"])
#print(gdf)

In [ ]:
print(output_file.active_geometry_name)
gdf.head()
#print(full_parms)

## ATL24 Request

atl24 request must be separate from main atl03 request. 

In [ ]:
parms = {
    "poly": srregion["poly"], #srextent,
    "t0": f'{temporal[0]:%Y-%m-%dT%H:%M:%SZ}',
    "t1": f'{temporal[1]:%Y-%m-%dT%H:%M:%SZ}',
    "srt": [0,1,2,3,4],
    "cnf": [-2, -1, 0, 1, 2, 3, 4],
    "atl24": {
        "compact": True,
        "confidence_threshold": 0.0,
        "class_ph": ["bathymetry"],
        "anc_fields": ["index_ph", "index_seg"]
    }
}



In [ ]:
gdfatl24 = sliderule.run("atl24x", parms)

In [ ]:
gdfatl24.info()
gdfatl24["class_ph"]
gdfatl24["index_ph"]

In [ ]:
gdfatl24.head()

## Gebco Request

In [ ]:
parms = {
    "poly": srregion["poly"], #srextent,
    "t0": f'{temporal[0]:%Y-%m-%dT%H:%M:%SZ}',
    "t1": f'{temporal[1]:%Y-%m-%dT%H:%M:%SZ}',
    "srt": [0,1,2,3,4],
    "cnf": [-2, -1, 0, 1, 2, 3, 4]
}
sliderule_req = DataRequest(spatial=srregion["poly"],
                    date_range=(f'{temporal[0]:%Y-%m-%d}', f'{temporal[1]:%Y-%m-%d}'),
                    download_dir= "./test_data/")
print(sliderule_req)
parms_sr = sliderule_req.build_atl03_params()

parms_sr["samples"] = {"gebco": {"asset": "gebco-s3"}}
print(parms)
print(parms_sr)

In [ ]:
parms_sr

In [ ]:
parms = {
    "poly": srregion["poly"], #srextent,
    "t0": f'{temporal[0]:%Y-%m-%dT%H:%M:%SZ}',
    "t1": f'{temporal[1]:%Y-%m-%dT%H:%M:%SZ}',
    "srt": [0,1,2,3,4],
    "cnf": [-2, -1, 0, 1, 2, 3, 4]
}
sliderule_req = DataRequest(spatial=srregion["poly"],
                    date_range=(f'{temporal[0]:%Y-%m-%d}', f'{temporal[1]:%Y-%m-%d}'),
                    download_dir= "./test_data/")
print(sliderule_req)
parms_sr = sliderule_req.build_atl03_params()

parms_sr["samples"] = {"gebco": {"asset": "gebco-s3"}}
print(parms)
print(parms_sr)

In [ ]:
rsps = sliderule.source("assets")

In [ ]:
parms

In [ ]:
gdf = sliderule.run("atl03x", parms_sr)

In [ ]:
gdf.info()

In [ ]:
gdf.head()

In [ ]:
gdf["index_ph"]
gdf[gdf["index_ph"] == 1418861]


In [ ]:
gdf.filter(regex='gebco')
#gdf["gebco.value"] < -23

In [ ]:

gdf.attrs

In [ ]:



filedir = gdf.attrs['file_directory']
filedir[gdf['mosaic.file_id'].iloc[0]]


sliderule.raster.sample("gebco-s3", srregion["poly"])

## ATL09


In [ ]:
parms = {
    "poly": srregion["poly"], #srextent,
    "t0": f'{temporal[0]:%Y-%m-%dT%H:%M:%SZ}',
    "t1": f'{temporal[1]:%Y-%m-%dT%H:%M:%SZ}',
    "srt": [0,1,2,3,4],
    "cnf": [-2, -1, 0, 1, 2, 3, 4]
}
sliderule_req = DataRequest(spatial=srregion["poly"],
                    date_range=(f'{temporal[0]:%Y-%m-%d}', f'{temporal[1]:%Y-%m-%d}'),
                    download_dir= "./test_data/")
print(sliderule_req)
parms_sr = sliderule_req.build_atl03_params()

parms_sr["samples"] = {"gebco": {"asset": "gebco-s3"}}
print(parms)
print(parms_sr)

In [ ]:
sliderule_req = DataRequest(spatial=srregion["poly"],
                    date_range=("2018-10-22", "2018-10-26"),
                    download_dir= "./test_data/")
parms_sr = sliderule_req.build_atl03_params()


In [ ]:
atl09_solar_radiation = {
            "atl09_fields": [
                "bckgrd_atlas/bckgrd_counts", 
                "bckgrd_atlas/bckgrd_counts_reduced",
                "bckgrd_atlas/bckgrd_rate",
                "low_rate/bsnow_con"],
        }


#parms_sr.update(atl09_solar_radiation)
parms_sr["atl09_fields"]

In [ ]:
gdf = sliderule.run("atl03x", parms_sr)

In [ ]:
gdf.head()

In [ ]:
gdf["bckgrd_atlas/bckgrd_rate"] == gdf["background_rate"]


In [ ]:
gdf["bckgrd_atlas/bckgrd_counts_reduced"] == gdf["background_rate"]

In [ ]:
gdf["bckgrd_atlas/bckgrd_counts"] == gdf["background_rate"]

In [ ]:
gdf["bckgrd_atlas/bckgrd_rate"] == gdf["bckgrd_rate"]

default bckgrd rate is NOT equal to atl09 backgrd rate

In [ ]:
gdf["bckgrd_atlas/bckgrd_counts_reduced"]

## ATL12

In [ ]:
import earthaccess
import xarray as xr

In [ ]:
auth = earthaccess.login()


In [ ]:
results = earthaccess.search_data(short_name="ATL12",
                                  version="007",
                                  cloud_hosted=True,
                                  temporal = ("2018-10-22","2018-10-23"),
                                  bounding_box = (-77.54, 36.66, -74.58, 39.63))
ds = earthaccess.open(results)
ds



## Raster sampling

In [ ]:
parms = {
    "poly": srregion["poly"], #srextent,
    "t0": f'{temporal[0]:%Y-%m-%dT%H:%M:%SZ}',
    "t1": f'{temporal[1]:%Y-%m-%dT%H:%M:%SZ}',
    "srt": [0,1,2,3,4],
    "cnf": [-2, -1, 0, 1, 2, 3, 4],
}

parms["samples"] = {"gebco": {"asset": "gebco-s3"}}

In [ ]:
from sliderule import earthdata

In [ ]:
region = [
    {"lon": -71.2, "lat": 41.5},
    {"lon": -70.8, "lat": 41.5},
    {"lon": -70.8, "lat": 42.0},
    {"lon": -71.2, "lat": 42.0},
    {"lon": -71.2, "lat": 41.5},
]

granules = sliderule.earthdata.cmr(
    short_name="ASTWBD",
    version="001",
    polygon=region,
    return_metadata=True,
)

print(granules)